In [1]:
#Importación de funciones de los diferentes archivos
import pandas as pd
import numpy as np

# Guardo todos los datos
df = pd.read_csv('data/clothDataset_36_.csv') 

In [2]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import torch
import os, os.path

class ClothDataset(Dataset):
    """
    Dataset con ANCHORING + features completas.

    Layout de features por vértice (16 total):
      [0-2]   x_anc, y_anc, z_anc        <- posición anclada (frame 0, fija)
      [3-5]   x,     y,     z             <- posición actual
      [6-8]   vx,    vy,    vz            <- velocidad
      [9]     sdf                         <- distancia signo a la esfera
      [10-12] nx,    ny,    nz            <- normal del vértice
      [13]    md                          <- max distance / distancia de reposo
      [14-15] u,     v                    <- coordenadas UV

    Target (output): posición absoluta en t+1  →  (V, 3)
    """

    # Features dinámicas del frame t (en el orden del CSV)
    DYNAMIC_PREFIXES = ['x', 'y', 'z', 'vx', 'vy', 'vz', 'sdf', 'nx', 'ny', 'nz', 'md', 'u', 'v']
    POSITION_PREFIXES = ['x', 'y', 'z']
    NUM_DYNAMIC = len(DYNAMIC_PREFIXES)   # 13
    NUM_ANCHOR  = 3
    NUM_FEATURES = NUM_ANCHOR + NUM_DYNAMIC  # 16

    def __init__(self, csv_data, num_vertices=6):
        self.data = csv_data
        self.data.columns = self.data.columns.str.strip()
        self.num_vertices = num_vertices

        # Posiciones de salida (frame t+1)
        self.output_positions = self.data.filter(regex=r'^[xyz]\d+$')
        self.output_positions = self.output_positions.iloc[1:]   # quitar primera fila
        self.data = self.data.iloc[:-1]                          # quitar última fila

        # Ancla: posiciones del frame 0 (constante durante toda la simulación)
        anchor_row = self.data.iloc[0]
        self.anchor = []
        for i in range(self.num_vertices):
            vtx_cols = [f"{p}{i}" for p in self.POSITION_PREFIXES]
            self.anchor.append(anchor_row[vtx_cols].values.astype('float32'))
        # (num_vertices, 3) — tensor fijo
        self.anchor_tensor = torch.tensor(self.anchor)

    def __len__(self):
        return len(self.data)

    def num_features(self):
        return self.NUM_FEATURES  # 16

    def num_vertex(self):
        return self.num_vertices

    def _get_frame_output_tensor(self, idx):
        """Posiciones absolutas en t+1 — target del modelo."""
        row = self.output_positions.iloc[idx]
        frame_data = []
        for i in range(self.num_vertices):
            vtx_cols = [f"{p}{i}" for p in self.POSITION_PREFIXES]
            frame_data.append(row[vtx_cols].values.astype('float32'))
        return torch.tensor(frame_data)  # (V, 3)

    def _get_frame_tensor(self, idx):
        """
        Construye el tensor de entrada con anchoring y features completas:
          [x_anc, y_anc, z_anc | x, y, z, vx, vy, vz, sdf, nx, ny, nz, md, u, v]
        """
        row = self.data.iloc[idx]
        frame_data = []
        for i in range(self.num_vertices):
            # Ancla (frame 0, inmutable)
            anc = self.anchor[i]  # (3,)
            # Features dinámicas del frame t
            dyn_cols = [f"{p}{i}" for p in self.DYNAMIC_PREFIXES]
            dyn = row[dyn_cols].values.astype('float32')  # (13,)
            # Concatenar: [anc | dyn]  → (16,)
            vertex_features = np.concatenate([anc, dyn])
            frame_data.append(vertex_features)
        return torch.tensor(frame_data)  # (V, 16)

    def __getitem__(self, idx):
        frame_t  = self._get_frame_tensor(idx)         # (V, 16)
        frame_t1 = self._get_frame_output_tensor(idx)  # (V, 3)
        return frame_t, frame_t1


# --- Uso ---
dataset    = ClothDataset(df)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

print(f"Features por vértice: {dataset.num_features()}")
print(f"  [0-2]  ancla xyz      (3)")
print(f"  [3-5]  pos xyz        (3)")
print(f"  [6-8]  vel xyz        (3)")
print(f"  [9]    sdf            (1)")
print(f"  [10-12] normal xyz    (3)")
print(f"  [13]   md             (1)")
print(f"  [14-15] uv            (2)")
for batch_data, batch_frames in dataloader:
    print(f"Input shape:  {batch_data.shape}")    # (B, V, 16)
    print(f"Target shape: {batch_frames.shape}")  # (B, V, 3)
    break


Features por vértice: 16
  [0-2]  ancla xyz      (3)
  [3-5]  pos xyz        (3)
  [6-8]  vel xyz        (3)
  [9]    sdf            (1)
  [10-12] normal xyz    (3)
  [13]   md             (1)
  [14-15] uv            (2)
Input shape:  torch.Size([4, 6, 16])
Target shape: torch.Size([4, 6, 3])


C:\Users\Eva\AppData\Local\Temp\ipykernel_29304\58445595.py:46: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\cb\pytorch_1000000000000\work\torch\csrc\utils\tensor_new.cpp:281.)
  self.anchor_tensor = torch.tensor(self.anchor)


In [3]:
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data

class MyModule(nn.Module):
    """
    Red neuronal con anchoring + features completas.
    Entrada: (Batch, V, 16)
      [x_anc, y_anc, z_anc | x, y, z, vx, vy, vz, sdf, nx, ny, nz, md, u, v]
    Salida:  (Batch, V, 3)
      desplazamiento Δ predicho → pos_t+1 = pos_t + Δ

    La ancla [0-2] permite a la red comparar siempre con el estado de reposo,
    eliminando la deriva acumulativa en simulaciones largas.
    """
    def __init__(self, num_verts=6, num_feats=16):
        super().__init__()
        self.input_dim  = num_verts * num_feats  # 6*16 = 96
        self.output_dim = num_verts * 3           # 6*3  = 18

        self.net = nn.Sequential(
            nn.Linear(self.input_dim, 1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, 1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, 1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, self.output_dim)
        )

    def forward(self, x):
        # x: (Batch, V, 16)
        x = x.view(x.size(0), -1)   # (Batch, 96)
        x = self.net(x)
        return x.view(x.size(0), -1, 3)  # (Batch, V, 3)


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import numpy as np
import json

# Layout del vector de entrada (16 features por vértice):
#   [0-2]   ancla xyz
#   [3-5]   pos xyz      ← pos_t usada para calcular Δ target
#   [6-8]   vel xyz
#   [9]     sdf
#   [10-12] normal xyz
#   [13]    md
#   [14-15] uv
NUM_FEATS   = 16
POS_T_SLICE = slice(3, 6)  # canales de la posición actual dentro del input

# --- Dataloaders ---
train_dataset = dataset
test_dataset  = dataset

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=4, shuffle=False)

# --- Estadísticas globales de normalización ---
all_inputs  = []
all_targets = []

for batch_t, batch_t1 in dataloader:
    # batch_t: (B, V, 16)
    all_inputs.append(batch_t.view(-1, NUM_FEATS))

    pos_t = batch_t[..., POS_T_SLICE]  # posición actual en canales 3-5
    delta = batch_t1 - pos_t
    all_targets.append(delta.view(-1, 3))

full_input_tensor  = torch.cat(all_inputs,  dim=0)
full_target_tensor = torch.cat(all_targets, dim=0)

input_mean  = full_input_tensor.mean(dim=0)
input_std   = full_input_tensor.std(dim=0)
target_mean = full_target_tensor.mean(dim=0)
target_std  = full_target_tensor.std(dim=0)

# Evitar división por cero (canales constantes como md o uv lo agradecen)
input_std  = input_std.clamp(min=1e-8)
target_std = target_std.clamp(min=1e-8)

norm_data = {
    "mean":        input_mean.tolist(),
    "std":         input_std.tolist(),
    "target_mean": target_mean.tolist(),
    "target_std":  target_std.tolist()
}
with open("cloth_norm_params_anchored_full.json", "w") as f:
    json.dump(norm_data, f)
print("Norm params guardados en cloth_norm_params_anchored_full.json")
print(f"  input_mean shape: {input_mean.shape}  (esperado: 16)")

# --- Modelo, loss y optimizador ---
model     = MyModule(num_verts=6, num_feats=NUM_FEATS)
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
print("Inicio del entrenamiento (anchoring + features completas)...")

# --- Bucle de entrenamiento ---
epochs = 50

for epoch in range(epochs):

    # TRAIN
    model.train()
    train_loss = 0.0

    for batch_t, batch_t1 in train_loader:
        batch_t  = batch_t.float()
        batch_t1 = batch_t1.float()

        batch_t_norm = (batch_t - input_mean) / input_std

        pos_t             = batch_t[..., POS_T_SLICE]
        target_delta      = batch_t1 - pos_t
        target_delta_norm = (target_delta - target_mean) / target_std

        pred_delta_norm = model(batch_t_norm)
        loss = criterion(pred_delta_norm, target_delta_norm)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # TEST / VALIDACIÓN
    model.eval()
    test_loss            = 0.0
    total_distance_error = 0.0
    total_vertices       = 0
    all_pred_deltas      = []
    all_target_deltas    = []

    with torch.no_grad():
        for batch_t, batch_t1 in test_loader:
            batch_t, batch_t1 = batch_t.float(), batch_t1.float()

            batch_t_norm    = (batch_t - input_mean) / input_std
            pred_delta_norm = model(batch_t_norm)

            pos_t             = batch_t[..., POS_T_SLICE]
            target_delta      = batch_t1 - pos_t
            target_delta_norm = (target_delta - target_mean) / target_std

            loss = criterion(pred_delta_norm, target_delta_norm)
            test_loss += loss.item()

            pred_delta_meters = pred_delta_norm * target_std + target_mean

            distances = torch.norm(pred_delta_meters - target_delta, dim=-1)
            total_distance_error += distances.sum().item()
            total_vertices       += distances.numel()

            all_pred_deltas.append(pred_delta_meters.cpu().numpy())
            all_target_deltas.append(target_delta.cpu().numpy())

    avg_test_loss      = test_loss  / len(test_loader)
    avg_distance_error = total_distance_error / total_vertices

    print(f'Epoch {epoch+1:03d} | Train: {avg_train_loss:.6f} | Test: {avg_test_loss:.6f} | Vertex Err: {avg_distance_error:.6f} m')

# Arrays finales para visualización
preds_pos_np   = np.concatenate(all_pred_deltas,  axis=0)
targets_pos_np = np.concatenate(all_target_deltas, axis=0)


Norm params guardados en cloth_norm_params_anchored_full.json
  input_mean shape: torch.Size([16])  (esperado: 16)
Inicio del entrenamiento (anchoring + features completas)...
Epoch 001 | Train: 0.172233 | Test: 0.115311 | Vertex Err: 0.001483 m


KeyboardInterrupt: 

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# 1. Combine the collected batches into single large arrays
preds_np = np.concatenate(all_pred_deltas, axis=0)     # Shape: (Total_Frames, Vertices, 3)
targets_np = np.concatenate(all_target_deltas, axis=0) # Shape: (Total_Frames, Vertices, 3)

# 2. Reshape to 2D: (Total_Frames, Vertices * 3)
preds_2d = preds_np.reshape(preds_np.shape[0], -1)
targets_2d = targets_np.reshape(targets_np.shape[0], -1)

# 3. Initialize PCA
pca = PCA(n_components=2) # 2 components for a 2D scatter plot

# CRITICAL: Fit the PCA on the TARGETS (Ground Truth) to define the coordinate space.
# If you fit two separate PCAs, the axes will mean different things and you cannot compare them!
pca.fit(targets_2d)

# Transform both datasets into this shared PCA space
pca_targets = pca.transform(targets_2d)
pca_preds = pca.transform(preds_2d)

# 4. Plot the comparison
plt.figure(figsize=(10, 8))

# Plot ground truth in blue
plt.scatter(pca_targets[:, 0], pca_targets[:, 1], alpha=0.5, label='Ground Truth (Targets)', color='blue', s=10)

# Plot predictions in red
plt.scatter(pca_preds[:, 0], pca_preds[:, 1], alpha=0.5, label='Model Predictions', color='red', s=10)

plt.title('PCA of Cloth Dynamics: Ground Truth vs Predictions')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Flatten the arrays so we just have a massive list of X, Y, and Z coordinates
true_flat = targets_pos_np.reshape(-1)
pred_flat = preds_pos_np.reshape(-1)

plt.figure(figsize=(12, 12))
plt.scatter(true_flat, pred_flat, alpha=0.1, color='purple', s=2)

# Draw the perfect prediction line (y = x)
min_val = min(true_flat.min(), pred_flat.min())
max_val = max(true_flat.max(), pred_flat.max())
plt.plot([min_val, max_val], [min_val, max_val], color='black', linestyle='--', linewidth=2, label='Perfect Prediction')

plt.title('Predicted vs. Real Positions (All Coordinates)')
plt.xlabel('Real Position Coordinate')
plt.ylabel('Predicted Position Coordinate')
plt.legend()
plt.grid(True)
plt.axis('equal') # Keeps the aspect ratio square
plt.show()

In [ ]:
# Choose a random frame from your test set to visualize
frame_idx = 200# Change this to look at different moments
num_vertices = targets_pos_np.shape[1]

real_frame = targets_pos_np[frame_idx] # Shape: (Vertices, 3)
pred_frame = preds_pos_np[frame_idx]   # Shape: (Vertices, 3)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Extract X, Y, Z for Real positions
rx, ry, rz = real_frame[:, 0], real_frame[:, 1], real_frame[:, 2]
# Extract X, Y, Z for Predicted positions
px, py, pz = pred_frame[:, 0], pred_frame[:, 1], pred_frame[:, 2]

# Plot the real cloth vertices in Blue
ax.scatter(rx, ry, rz, c='blue', label='Real Shape', s=50, marker='o')

# Plot the predicted cloth vertices in Red
ax.scatter(px, py, pz, c='red', label='Predicted Shape', s=50, marker='x')

# Draw lines connecting the predicted vertex to the real vertex to show the error
for i in range(num_vertices):
    ax.plot([rx[i], px[i]], [ry[i], py[i]], [rz[i], pz[i]], color='gray', linestyle='dotted')

ax.set_title(f'Cloth Shape Comparison (Frame {frame_idx})')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.legend()
plt.show()

In [ ]:
import torch
import onnx
import onnxruntime

model.eval()

# Input: (Batch=1, Vertices=6, Features=16)
# Layout por vértice: [x_anc, y_anc, z_anc | x, y, z, vx, vy, vz, sdf, nx, ny, nz, md, u, v]
dummy_input = torch.randn(1, 6, 16)

torch.onnx.export(
    model,
    dummy_input,
    "cloth_anchoring_full.onnx",
    export_params=True,
    opset_version=9,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

print("Exportado a cloth_anchoring_full.onnx")
print("Input shape en Unity/Sentis: (batch, 6, 16)")
print("Orden de canales por vértice:")
print("  [0-2]  ancla xyz       (frame 0, constante)")
print("  [3-5]  pos xyz         (posición actual)")
print("  [6-8]  vel xyz         (velocidad)")
print("  [9]    sdf             (distancia a la esfera)")
print("  [10-12] normal xyz     (normal del vértice)")
print("  [13]   md              (max distance)")
print("  [14-15] uv             (coordenadas UV)")

def test_parity(pytorch_model, onnx_path, input_shape=(1, 3, 224, 224)):
    # 1. Preparar el modelo de PyTorch
    pytorch_model.eval()
    
    # 2. Crear un input aleatorio (Dummy Input)
    dummy_input = torch.randn(*input_shape)
    
    # 3. Obtener inferencia de PyTorch
    with torch.no_grad():
        torch_output = pytorch_model(dummy_input).numpy()
    
    # 4. Preparar la sesión de ONNX Runtime
    ort_session = ort.InferenceSession(onnx_path)
    
    # 5. Obtener inferencia de ONNX
    # Necesitamos pasar el input como un diccionario de numpy
    input_name = ort_session.get_inputs()[0].name
    onnx_inputs = {input_name: dummy_input.numpy()}
    onnx_output = ort_session.run(None, onnx_inputs)[0]
    
    # 6. Comparar resultados (Test de Paridad)
    try:
        # rtol=1e-03 y atol=1e-05 son valores estándar para float32
        np.testing.assert_allclose(torch_output, onnx_output, rtol=1e-03, atol=1e-05)
        print(" ¡Éxito! Los resultados son idénticos dentro de la tolerancia.")
    except AssertionError as e:
        print("¡Error de Paridad! Los modelos producen resultados diferentes.")
        print(e)

# Ejemplo de uso:
test_parity(model, "cloth_anchoring_full.onnx")


In [ ]:

def test_parity(pytorch_model, onnx_path, input_shape=(1, 3, 224, 224)):
    # 1. Preparar el modelo de PyTorch
    pytorch_model.eval()
    
    # 2. Crear un input aleatorio (Dummy Input)
    dummy_input = torch.randn(*input_shape)
    
    # 3. Obtener inferencia de PyTorch
    with torch.no_grad():
        torch_output = pytorch_model(dummy_input).numpy()
    
    # 4. Preparar la sesión de ONNX Runtime
    ort_session = ort.InferenceSession(onnx_path)
    
    # 5. Obtener inferencia de ONNX
    # Necesitamos pasar el input como un diccionario de numpy
    input_name = ort_session.get_inputs()[0].name
    onnx_inputs = {input_name: dummy_input.numpy()}
    onnx_output = ort_session.run(None, onnx_inputs)[0]
    
    # 6. Comparar resultados (Test de Paridad)
    try:
        # rtol=1e-03 y atol=1e-05 son valores estándar para float32
        np.testing.assert_allclose(torch_output, onnx_output, rtol=1e-03, atol=1e-05)
        print(" ¡Éxito! Los resultados son idénticos dentro de la tolerancia.")
    except AssertionError as e:
        print("¡Error de Paridad! Los modelos producen resultados diferentes.")
        print(e)

# Ejemplo de uso:
test_parity(model, "cloth_anchoring_full.onnx")